In [1]:
from google.colab import drive

drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [2]:
import os

PROJECT_PATH = "/content/drive/MyDrive/Meridian_Churn_Capstone"
CLEAN_DATA_PATH = os.path.join(PROJECT_PATH, "Data", "Cleaned")

print("Project folder exists:", os.path.isdir(PROJECT_PATH))
print("Cleaned data folder exists:", os.path.isdir(CLEAN_DATA_PATH))

if os.path.isdir(CLEAN_DATA_PATH):
    print("\nFiles found:")
    print(os.listdir(CLEAN_DATA_PATH))

Project folder exists: True
Cleaned data folder exists: True

Files found:
['customers.csv', 'products.csv', 'orders.csv', 'order_items.csv', 'support_tickets.csv']


In [3]:
# STEP 7.1A: Import libraries and connect Google Drive

import os
import pandas as pd
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

# Folder containing the cleaned datasets from Phase 6
CLEAN_DATA_PATH = (
    "/content/drive/MyDrive/"
    "Meridian_Churn_Capstone/Data/Cleaned"
)

# Confirm that the folder exists
print("Cleaned data folder exists:", os.path.isdir(CLEAN_DATA_PATH))
print("\nFiles in the cleaned data folder:")

if os.path.isdir(CLEAN_DATA_PATH):
    print(sorted(os.listdir(CLEAN_DATA_PATH)))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cleaned data folder exists: True

Files in the cleaned data folder:
['customers.csv', 'order_items.csv', 'orders.csv', 'products.csv', 'support_tickets.csv']


In [6]:
customers = pd.read_csv(os.path.join(CLEAN_DATA_PATH, "customers.csv"))
products = pd.read_csv(os.path.join(CLEAN_DATA_PATH, "products.csv"))
orders = pd.read_csv(os.path.join(CLEAN_DATA_PATH, "orders.csv"))
order_items = pd.read_csv(os.path.join(CLEAN_DATA_PATH, "order_items.csv"))
support_tickets = pd.read_csv(
    os.path.join(CLEAN_DATA_PATH, "support_tickets.csv")
)

print("Cleaned datasets loaded successfully.\n")

for name, df in {
    "Customers": customers,
    "Products": products,
    "Orders": orders,
    "Order Items": order_items,
    "Support Tickets": support_tickets
}.items():
    print(f"{name}: {df.shape[0]:,} rows × {df.shape[1]} columns")

Cleaned datasets loaded successfully.

Customers: 6,090 rows × 22 columns
Products: 220 rows × 11 columns
Orders: 28,272 rows × 12 columns
Order Items: 73,465 rows × 7 columns
Support Tickets: 5,200 rows × 10 columns


In [ ]:
import numpy as np

PROJECT_PATH = "/content/drive/MyDrive/Meridian_Churn_Capstone"
CLEAN_DATA_PATH = os.path.join(PROJECT_PATH, "Data", "Cleaned")

# Confirm that the cleaned-data folder exists
if not os.path.isdir(CLEAN_DATA_PATH):
    raise FileNotFoundError(
        f"Cleaned data folder not found: {CLEAN_DATA_PATH}"
    )

# Load all five datasets
customers = pd.read_csv(
    os.path.join(CLEAN_DATA_PATH, "customers.csv")
)
products = pd.read_csv(
    os.path.join(CLEAN_DATA_PATH, "products.csv")
)
orders = pd.read_csv(
    os.path.join(CLEAN_DATA_PATH, "orders.csv")
)
order_items = pd.read_csv(
    os.path.join(CLEAN_DATA_PATH, "order_items.csv")
)
support_tickets = pd.read_csv(
    os.path.join(CLEAN_DATA_PATH, "support_tickets.csv")
)

# Display verified dimensions
datasets = {
    "Customers": customers,
    "Products": products,
    "Orders": orders,
    "Order Items": order_items,
    "Support Tickets": support_tickets
}

print("All five datasets loaded successfully.\n")

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]:,} rows × {df.shape[1]} columns")

All five datasets loaded successfully.

Customers: 6,090 rows × 22 columns
Products: 220 rows × 11 columns
Orders: 28,272 rows × 12 columns
Order Items: 73,465 rows × 7 columns
Support Tickets: 5,200 rows × 10 columns


In [7]:
# Phase 7.2: Define customer churn

SNAPSHOT_DATE = pd.Timestamp("2026-07-01")
WINDOW_START = SNAPSHOT_DATE - pd.Timedelta(days=180)

print("Snapshot date:", SNAPSHOT_DATE.date())
print("180-day window starts:", WINDOW_START.date())
print("Window ends:", SNAPSHOT_DATE.date())

# Check the actual column names and order statuses first
print("\nOrder columns:")
print(orders.columns.tolist())

print("\nOrder status counts:")
print(orders["order_status"].value_counts(dropna=False))

print("\nMissing order dates:", orders["order_date"].isna().sum())


Snapshot date: 2026-07-01
180-day window starts: 2026-01-02
Window ends: 2026-07-01

Order columns:
['order_id', 'customer_id', 'order_date', 'channel', 'order_status', 'payment_method', 'shipping_cost', 'discount_pct', 'total_amount', 'order_date_missing', 'negative_shipping_cost_invalid', 'customer_reference_missing']

Order status counts:
order_status
Completed    25481
Returned      1230
Cancelled      976
Pending        585
Name: count, dtype: int64

Missing order dates: 424


In [10]:
# Phase 7.2: Create the customer churn target

SNAPSHOT_DATE = pd.Timestamp("2026-07-01")
WINDOW_START = SNAPSHOT_DATE - pd.Timedelta(days=180)

# Convert order dates to datetime safely
orders["order_date"] = pd.to_datetime(
    orders["order_date"], errors="coerce"
)

# Keep only completed orders within the 180-day window
completed_in_window = orders[
    (orders["order_status"].str.strip().str.lower() == "completed")
    & (orders["order_date"] >= WINDOW_START)
    & (orders["order_date"] < SNAPSHOT_DATE + pd.Timedelta(days=1))
].copy()

# Exclude orders without a customer ID
completed_in_window = completed_in_window[
    completed_in_window["customer_id"].notna()
]

# Count completed orders for each customer
recent_order_counts = (
    completed_in_window
    .groupby("customer_id")["order_id"]
    .nunique()
    .rename("completed_orders_last_180_days")
)

# Start with every customer in the Customers dataset
customer_analysis = customers.copy()

customer_analysis = customer_analysis.merge(
    recent_order_counts,
    left_on="customer_id",
    right_index=True,
    how="left"
)

customer_analysis["completed_orders_last_180_days"] = (
    customer_analysis["completed_orders_last_180_days"]
    .fillna(0)
    .astype(int)
)

# Churn target: 1 = churned; 0 = active
customer_analysis["churn"] = (
    customer_analysis["completed_orders_last_180_days"] == 0
).astype(int)

# Summarise the result
total_customers = len(customer_analysis)
churned_customers = int(customer_analysis["churn"].sum())
active_customers = total_customers - churned_customers
churn_rate = (
    churned_customers / total_customers * 100
    if total_customers else 0
)

print("Customer churn target created successfully.")
print(f"Total customers: {total_customers:,}")
print(f"Churned customers: {churned_customers:,}")
print(f"Active customers: {active_customers:,}")
print(f"Overall churn rate: {churn_rate:.2f}%")
print("\nChurn label counts:")
print(customer_analysis["churn"].value_counts().sort_index())

Customer churn target created successfully.
Total customers: 6,090
Churned customers: 4,072
Active customers: 2,018
Overall churn rate: 66.86%

Churn label counts:
churn
0    2018
1    4072
Name: count, dtype: int64


In [11]:
# Phase 7.3: Churn analysis by customer segment

import pandas as pd
import numpy as np

# Work from the customer-level churn dataset
segment_data = customer_analysis.copy()

# Ensure signup dates are parsed correctly
segment_data["signup_date"] = pd.to_datetime(
    segment_data["signup_date"], errors="coerce"
)

# Make missing categories visible rather than silently dropping them
segment_columns = [
    "membership_tier",
    "preferred_channel",
    "state",
    "referral_source"
]

for col in segment_columns:
    segment_data[col] = (
        segment_data[col]
        .fillna("Unknown / Not recorded")
        .astype(str)
        .str.strip()
        .replace("", "Unknown / Not recorded")
    )

# Create signup-year cohorts
segment_data["signup_cohort"] = (
    segment_data["signup_date"].dt.year.astype("Int64").astype(str)
)
segment_data.loc[
    segment_data["signup_date"].isna(), "signup_cohort"
] = "Unknown / Not recorded"

# Summarise customer count and churn rate for each segment
def churn_summary(data, segment_col):
    result = (
        data.groupby(segment_col, dropna=False)
        .agg(
            customers=("customer_id", "nunique"),
            churned_customers=("churn", "sum"),
            churn_rate=("churn", "mean")
        )
        .reset_index()
    )

    result["churn_rate_pct"] = result["churn_rate"] * 100

    return result.drop(columns="churn").sort_values(
        "churn_rate_pct", ascending=False
    ).round({"churn_rate_pct": 2})

segment_results = {
    "Membership tier": churn_summary(segment_data, "membership_tier"),
    "Preferred channel": churn_summary(segment_data, "preferred_channel"),
    "State": churn_summary(segment_data, "state"),
    "Referral source": churn_summary(segment_data, "referral_source"),
    "Signup cohort": churn_summary(segment_data, "signup_cohort")
}

for title, table in segment_results.items():
    print("\n" + "=" * 65)
    print(title.upper())
    print("=" * 65)
    print(table.to_string(index=False))


KeyError: "['churn'] not found in axis"

In [12]:
# Corrected Phase 7.3 churn summary function

def churn_summary(data, segment_col):
    result = (
        data.groupby(segment_col, dropna=False)
        .agg(
            customers=("customer_id", "nunique"),
            churned_customers=("churn", "sum"),
            churn_rate=("churn", "mean")
        )
        .reset_index()
    )

    result["churn_rate_pct"] = result["churn_rate"] * 100

    result = result.drop(columns=["churn_rate"])

    return (
        result
        .sort_values("churn_rate_pct", ascending=False)
        .round({"churn_rate_pct": 2})
    )


# Recalculate all five segment summaries
segment_results = {
    "Membership tier": churn_summary(
        segment_data, "membership_tier"
    ),
    "Preferred channel": churn_summary(
        segment_data, "preferred_channel"
    ),
    "State": churn_summary(
        segment_data, "state"
    ),
    "Referral source": churn_summary(
        segment_data, "referral_source"
    ),
    "Signup cohort": churn_summary(
        segment_data, "signup_cohort"
    )
}

# Display results
for title, table in segment_results.items():
    print("\n" + "=" * 65)
    print(title.upper())
    print("=" * 65)
    print(table.to_string(index=False))



MEMBERSHIP TIER
membership_tier  customers  churned_customers  churn_rate_pct
           Gold       1126                765           67.94
         Silver       1768               1184           66.97
         Bronze       2785               1851           66.46
       Platinum        411                272           66.18

PREFERRED CHANNEL
preferred_channel  customers  churned_customers  churn_rate_pct
         In-Store       2041               1387           67.96
       Mobile App       2053               1368           66.63
           Online       1996               1317           65.98

STATE
         state  customers  churned_customers  churn_rate_pct
            Wi        138                101           73.19
            Mi        149                109           73.15
            Tn        144                104           72.22
            Nj        146                105           71.92
            Oh        164                117           71.34
            Az        165

In [13]:
# Phase 7.3: Standardize state names before geographic analysis

state_map = {
    "AZ": "Arizona",
    "CA": "California",
    "FL": "Florida",
    "GA": "Georgia",
    "IL": "Illinois",
    "IN": "Indiana",
    "MA": "Massachusetts",
    "MD": "Maryland",
    "MI": "Michigan",
    "MO": "Missouri",
    "NC": "North Carolina",
    "NJ": "New Jersey",
    "NY": "New York",
    "OH": "Ohio",
    "PA": "Pennsylvania",
    "TN": "Tennessee",
    "TX": "Texas",
    "VA": "Virginia",
    "WA": "Washington",
    "WI": "Wisconsin"
}

segment_data["state_standardized"] = (
    segment_data["state"]
    .fillna("Unknown / Not recorded")
    .astype(str)
    .str.strip()
    .apply(
        lambda value: state_map.get(value.upper(), value)
        if value != "Unknown / Not recorded"
        else value
    )
)

state_results = churn_summary(
    segment_data,
    "state_standardized"
)

print("STATE CHURN RATES AFTER STANDARDIZATION")
print(state_results.to_string(index=False))

print("\nNumber of distinct state labels before:",
      segment_data["state"].nunique())
print("Number of distinct state labels after:",
      segment_data["state_standardized"].nunique())

STATE CHURN RATES AFTER STANDARDIZATION
state_standardized  customers  churned_customers  churn_rate_pct
          Michigan        294                208           70.75
         Wisconsin        306                215           70.26
             Texas        298                207           69.46
          Illinois        314                215           68.47
              Ohio        292                199           68.15
         Tennessee        302                205           67.88
          Virginia        300                203           67.67
        New Jersey        292                197           67.47
        California        316                213           67.41
           Georgia        324                217           66.98
          Maryland        296                198           66.89
    North Carolina        266                177           66.54
          Missouri        326                215           65.95
           Arizona        321                211  

In [14]:
# Phase 7.3: Standardize state names before geographic analysis

state_map = {
    "AZ": "Arizona",
    "CA": "California",
    "FL": "Florida",
    "GA": "Georgia",
    "IL": "Illinois",
    "IN": "Indiana",
    "MA": "Massachusetts",
    "MD": "Maryland",
    "MI": "Michigan",
    "MO": "Missouri",
    "NC": "North Carolina",
    "NJ": "New Jersey",
    "NY": "New York",
    "OH": "Ohio",
    "PA": "Pennsylvania",
    "TN": "Tennessee",
    "TX": "Texas",
    "VA": "Virginia",
    "WA": "Washington",
    "WI": "Wisconsin"
}

segment_data["state_standardized"] = (
    segment_data["state"]
    .fillna("Unknown / Not recorded")
    .astype(str)
    .str.strip()
    .apply(
        lambda value: state_map.get(value.upper(), value)
        if value != "Unknown / Not recorded"
        else value
    )
)

state_results = churn_summary(
    segment_data,
    "state_standardized"
)

print("STATE CHURN RATES AFTER STANDARDIZATION")
print(state_results.to_string(index=False))

print("\nNumber of distinct state labels before:",
      segment_data["state"].nunique())
print("Number of distinct state labels after:",
      segment_data["state_standardized"].nunique())

STATE CHURN RATES AFTER STANDARDIZATION
state_standardized  customers  churned_customers  churn_rate_pct
          Michigan        294                208           70.75
         Wisconsin        306                215           70.26
             Texas        298                207           69.46
          Illinois        314                215           68.47
              Ohio        292                199           68.15
         Tennessee        302                205           67.88
          Virginia        300                203           67.67
        New Jersey        292                197           67.47
        California        316                213           67.41
           Georgia        324                217           66.98
          Maryland        296                198           66.89
    North Carolina        266                177           66.54
          Missouri        326                215           65.95
           Arizona        321                211  

In [15]:
# Phase 7.4: Recalculate churn consistently and engineer RFM features

SNAPSHOT_DATE = pd.Timestamp("2026-07-01")
WINDOW_START = SNAPSHOT_DATE - pd.Timedelta(days=180)

orders["order_date"] = pd.to_datetime(
    orders["order_date"], errors="coerce"
)

# Completed orders with valid dates before the snapshot date
completed_valid = orders[
    (orders["order_status"].str.strip().str.lower() == "completed")
    & orders["order_date"].notna()
    & (orders["order_date"] < SNAPSHOT_DATE)
    & orders["customer_id"].notna()
].copy()

# Completed orders in the strict 180-day churn window:
# 2 January inclusive, 1 July exclusive
completed_180d = completed_valid[
    completed_valid["order_date"] >= WINDOW_START
].copy()

# Count distinct orders in the 180-day window
recent_counts = (
    completed_180d.groupby("customer_id")["order_id"]
    .nunique()
    .rename("completed_orders_last_180_days")
)

# Rebuild the customer-level churn dataset
customer_analysis = customers.copy()

customer_analysis = customer_analysis.merge(
    recent_counts,
    left_on="customer_id",
    right_index=True,
    how="left"
)

customer_analysis["completed_orders_last_180_days"] = (
    customer_analysis["completed_orders_last_180_days"]
    .fillna(0)
    .astype(int)
)

customer_analysis["churn"] = (
    customer_analysis["completed_orders_last_180_days"] == 0
).astype(int)

# Calculate RFM using completed orders before the snapshot date.
# Monetary value uses order-level totals to avoid double-counting
# orders when order items are joined.
rfm = (
    completed_valid.groupby("customer_id")
    .agg(
        last_completed_order=("order_date", "max"),
        frequency=("order_id", "nunique"),
        monetary=("total_amount", "sum")
    )
    .reset_index()
)

rfm["recency_days"] = (
    SNAPSHOT_DATE - rfm["last_completed_order"]
).dt.days

rfm = rfm[
    ["customer_id", "recency_days", "frequency", "monetary"]
]

# Attach RFM features to every customer
customer_analysis = customer_analysis.merge(
    rfm,
    on="customer_id",
    how="left"
)

# Customers without a valid completed order have no recency;
# frequency and monetary value are set to zero.
customer_analysis["frequency"] = (
    customer_analysis["frequency"].fillna(0).astype(int)
)

customer_analysis["monetary"] = (
    customer_analysis["monetary"].fillna(0)
)

# Report refreshed churn and RFM results
print("CHURN RESULTS USING THE STRICT 180-DAY WINDOW")
print("Snapshot date:", SNAPSHOT_DATE.date())
print("Window start (inclusive):", WINDOW_START.date())
print("Window end (exclusive):", SNAPSHOT_DATE.date())
print("Total customers:", len(customer_analysis))
print("Churned customers:", int(customer_analysis["churn"].sum()))
print("Active customers:", int((customer_analysis["churn"] == 0).sum()))
print(
    "Churn rate:",
    f"{customer_analysis['churn'].mean() * 100:.2f}%"
)

print("\nRFM FEATURE CHECK")
print(
    customer_analysis[
        ["recency_days", "frequency", "monetary"]
    ].describe().round(2)
)

print("\nCustomers with no valid completed orders:",
      int(customer_analysis["frequency"].eq(0).sum()))

print("\nSample of customer RFM features:")
print(
    customer_analysis[
        ["customer_id", "churn", "recency_days", "frequency", "monetary"]
    ].head(10).to_string(index=False)
)

CHURN RESULTS USING THE STRICT 180-DAY WINDOW
Snapshot date: 2026-07-01
Window start (inclusive): 2026-01-02
Window end (exclusive): 2026-07-01
Total customers: 6090
Churned customers: 4075
Active customers: 2015
Churn rate: 66.91%

RFM FEATURE CHECK
       recency_days  frequency  monetary
count       5640.00    6090.00   6090.00
mean         363.38       4.12   6334.56
std          297.27       2.15   4088.57
min            1.00       0.00      0.00
25%          115.00       3.00   3376.74
50%          301.00       4.00   5942.24
75%          541.00       6.00   8811.88
max         2115.00      14.00  35395.37

Customers with no valid completed orders: 450

Sample of customer RFM features:
customer_id  churn  recency_days  frequency  monetary
 CUST002565      0          30.0          7   9149.70
 CUST004275      1           NaN          0      0.00
 CUST003045      1         664.0          3   8742.62
 CUST005168      1         186.0          4   9395.18
 CUST003986      1         64

In [16]:
# Phase 7.5: Engineer customer support features

support_tickets["ticket_date"] = pd.to_datetime(
    support_tickets["ticket_date"], errors="coerce"
)

# Use tickets recorded before the snapshot date
tickets_before_snapshot = support_tickets[
    support_tickets["ticket_date"].notna()
    & (support_tickets["ticket_date"] < SNAPSHOT_DATE)
    & support_tickets["customer_id"].notna()
].copy()

# Aggregate support activity to one row per customer
support_features = (
    tickets_before_snapshot
    .groupby("customer_id")
    .agg(
        ticket_count=("ticket_id", "nunique"),
        avg_satisfaction=("satisfaction_score", "mean"),
        avg_resolution_hours=("resolution_time_hours", "mean")
    )
    .reset_index()
)

# Merge support features into the customer-level dataset
customer_analysis = customer_analysis.merge(
    support_features,
    on="customer_id",
    how="left"
)

# No recorded tickets means zero tickets.
# Satisfaction and resolution remain missing when no values exist.
customer_analysis["ticket_count"] = (
    customer_analysis["ticket_count"].fillna(0).astype(int)
)

print("SUPPORT FEATURES CREATED")
print("Customers analysed:", len(customer_analysis))
print("Customers with at least one recorded ticket:",
      int(customer_analysis["ticket_count"].gt(0).sum()))
print("Customers with no recorded tickets:",
      int(customer_analysis["ticket_count"].eq(0).sum()))

print("\nSUPPORT FEATURE SUMMARY")
print(
    customer_analysis[
        ["ticket_count", "avg_satisfaction", "avg_resolution_hours"]
    ].describe().round(2)
)

SUPPORT FEATURES CREATED
Customers analysed: 6090
Customers with at least one recorded ticket: 3529
Customers with no recorded tickets: 2561

SUPPORT FEATURE SUMMARY
       ticket_count  avg_satisfaction  avg_resolution_hours
count       6090.00           3247.00               3527.00
mean           0.85              3.88                 18.31
std            0.92              1.06                 21.70
min            0.00              1.00                  0.00
25%            0.00              3.00                  6.60
50%            1.00              4.00                 13.50
75%            1.00              5.00                 24.42
max            6.00              5.00                682.00


In [17]:
# Compare support features by churn status

support_by_churn = (
    customer_analysis
    .groupby("churn")
    .agg(
        customers=("customer_id", "nunique"),
        avg_ticket_count=("ticket_count", "mean"),
        median_ticket_count=("ticket_count", "median"),
        customers_with_tickets=(
            "ticket_count", lambda x: x.gt(0).sum()
        ),
        avg_satisfaction=("avg_satisfaction", "mean"),
        avg_resolution_hours=("avg_resolution_hours", "mean")
    )
    .reset_index()
)

support_by_churn["churn_group"] = support_by_churn["churn"].map(
    {0: "Active", 1: "Churned"}
)

support_by_churn = support_by_churn[
    [
        "churn_group",
        "customers",
        "avg_ticket_count",
        "median_ticket_count",
        "customers_with_tickets",
        "avg_satisfaction",
        "avg_resolution_hours"
    ]
]

print("SUPPORT EXPERIENCE BY CHURN STATUS")
print(support_by_churn.round(2).to_string(index=False))


SUPPORT EXPERIENCE BY CHURN STATUS
churn_group  customers  avg_ticket_count  median_ticket_count  customers_with_tickets  avg_satisfaction  avg_resolution_hours
     Active       2015              0.83                  1.0                    1162              3.92                 17.33
    Churned       4075              0.87                  1.0                    2367              3.87                 18.79


In [18]:
# Phase 7.6: RFM analysis by churn status

rfm_by_churn = (
    customer_analysis
    .groupby("churn")
    .agg(
        customers=("customer_id", "nunique"),
        avg_recency_days=("recency_days", "mean"),
        median_recency_days=("recency_days", "median"),
        avg_frequency=("frequency", "mean"),
        median_frequency=("frequency", "median"),
        avg_monetary=("monetary", "mean"),
        median_monetary=("monetary", "median"),
        customers_with_no_orders=(
            "frequency", lambda x: x.eq(0).sum()
        )
    )
    .reset_index()
)

rfm_by_churn["customer_group"] = rfm_by_churn["churn"].map(
    {0: "Active", 1: "Churned"}
)

rfm_by_churn = rfm_by_churn[
    [
        "customer_group",
        "customers",
        "avg_recency_days",
        "median_recency_days",
        "avg_frequency",
        "median_frequency",
        "avg_monetary",
        "median_monetary",
        "customers_with_no_orders"
    ]
]

print("RFM COMPARISON BY CHURN STATUS")
print(rfm_by_churn.round(2).to_string(index=False))


RFM COMPARISON BY CHURN STATUS
customer_group  customers  avg_recency_days  median_recency_days  avg_frequency  median_frequency  avg_monetary  median_monetary  customers_with_no_orders
        Active       2015             89.33                 87.0           4.80               5.0       7430.43          6990.46                         0
       Churned       4075            515.71                454.0           3.78               4.0       5792.67          5421.09                       450


Interpreting your RFM results
RFM measure	Active customers	Churned customers	Interpretation
Customers	2,015	4,075	About two-thirds of customers are classified as churned.
Average recency	89.33 days	515.71 days	Churned customers' last completed purchase was much longer ago.
Median recency	87 days	454 days	The typical churned customer has been inactive far longer.
Average frequency	4.80 orders	3.78 orders	Active customers have more completed orders on average.
Median frequency	5 orders	4 orders	The typical active customer has one more completed order.
Average monetary value	7,430.43	5,792.67	Active customers have higher average cumulative spend.
Median monetary value	6,990.46	5,421.09	The typical active customer has spent more overall.
Customers with no completed orders	0	450	All 450 customers with no valid completed orders are classified as churned.

What the results tell us
Finding 1 — Recency sharply distinguishes the two groups. The average recency for churned customers is approximately 516 days, compared with 89 days for active customers. This is expected under our 180-day churn definition: customers without a completed order in the recent window are labelled churned. Therefore, recency is valuable for describing churn, but it would cause target leakage if used to predict this same churn label at the same snapshot date.

Finding 2 — Active customers purchase more frequently. Active customers average 4.80 completed orders, compared with 3.78 for churned customers. This suggests that repeat-purchase behaviour is associated with remaining active, although this comparison alone does not establish causation.

Finding 3 — Active customers have higher cumulative spending. Their average total spend is 7,430.43, compared with 5,792.67 for churned customers. The median comparison shows the same pattern, so the difference is not explained solely by a few very high-spending customers.

Finding 4 — A substantial group has never completed an order. There are 450 such customers, representing approximately 7.4% of the customer base. They are all classified as churned under the current definition, so we should distinguish never-purchasers from customers who purchased previously but later became inactive when developing retention recommendations.

Finding 5 — The median results reinforce the overall pattern. The median active customer has five completed orders and cumulative spend of 6,990.46, compared with four orders and 5,421.09 for the median churned customer. This strengthens the descriptive evidence that activity, repeat purchasing, and cumulative spend are related.

In [19]:
# PHASE 7: PRODUCT CATEGORY AND CUSTOMER LOYALTY

# 1. Select valid completed orders before the snapshot date
completed_order_headers = orders[
    (orders["order_status"].astype(str).str.strip().str.lower() == "completed")
    & orders["order_date"].notna()
    & (orders["order_date"] < SNAPSHOT_DATE)
    & orders["customer_id"].isin(customers["customer_id"])
].copy()

# 2. Connect order items to their orders and product categories
category_items = (
    order_items
    .merge(
        completed_order_headers[["order_id", "customer_id"]],
        on="order_id",
        how="inner"
    )
    .merge(
        products[["product_id", "category"]],
        on="product_id",
        how="left"
    )
)

# 3. Calculate line-item spend where quantity and price are valid
category_items["item_spend"] = (
    category_items["quantity"] *
    category_items["unit_price_at_purchase"]
)

valid_category_items = category_items[
    category_items["category"].notna()
    & category_items["quantity"].notna()
    & (category_items["quantity"] > 0)
    & category_items["unit_price_at_purchase"].notna()
    & (category_items["unit_price_at_purchase"] >= 0)
].copy()

# 4. Summarise each customer's purchasing behaviour by category
customer_category = (
    valid_category_items
    .groupby(["category", "customer_id"])
    .agg(
        completed_orders=("order_id", "nunique"),
        units_purchased=("quantity", "sum"),
        category_item_spend=("item_spend", "sum")
    )
    .reset_index()
)

# 5. Attach the customer's churn status
customer_category = customer_category.merge(
    customer_analysis[["customer_id", "churn"]],
    on="customer_id",
    how="left"
)

# 6. Summarise category-level customer loyalty and value
category_summary = (
    customer_category
    .groupby("category")
    .agg(
        customers=("customer_id", "nunique"),
        customer_order_counts=("completed_orders", "sum"),
        repeat_customers=("completed_orders", lambda x: (x >= 2).sum()),
        avg_item_spend_per_customer=("category_item_spend", "mean"),
        median_item_spend_per_customer=("category_item_spend", "median"),
        churned_customers=("churn", "sum")
    )
    .reset_index()
)

category_summary["repeat_customer_pct"] = (
    category_summary["repeat_customers"]
    / category_summary["customers"] * 100
)

category_summary["churn_rate_pct"] = (
    category_summary["churned_customers"]
    / category_summary["customers"] * 100
)

category_summary = category_summary.sort_values(
    "avg_item_spend_per_customer",
    ascending=False
)

print("PRODUCT CATEGORY: CUSTOMER VALUE AND LOYALTY")
print(category_summary.round(2).to_string(index=False))

PRODUCT CATEGORY: CUSTOMER VALUE AND LOYALTY
              category  customers  customer_order_counts  repeat_customers  avg_item_spend_per_customer  median_item_spend_per_customer  churned_customers  repeat_customer_pct  churn_rate_pct
           Electronics       4946                  11128              3341                      1890.07                         1463.56               3154                67.55           63.77
Beauty & Personal Care       4847                  10502              3166                      1796.22                         1406.99               3067                65.32           63.28
        Home & Kitchen       4862                  10654              3252                      1522.17                         1227.16               3069                66.89           63.12
     Sports & Outdoors       4895                  10408              3147                      1449.93                         1124.97               3108                64.29           6

In [20]:
# PHASE 7: RECALCULATE CHURN BY CUSTOMER SEGMENT

import pandas as pd

segment_data = customer_analysis.copy()

# Standardise state abbreviations and full names
state_map = {
    "AZ": "Arizona", "CA": "California", "FL": "Florida",
    "GA": "Georgia", "IL": "Illinois", "IN": "Indiana",
    "MA": "Massachusetts", "MD": "Maryland", "MI": "Michigan",
    "MO": "Missouri", "NC": "North Carolina", "NJ": "New Jersey",
    "NY": "New York", "OH": "Ohio", "PA": "Pennsylvania",
    "TN": "Tennessee", "TX": "Texas", "VA": "Virginia",
    "WA": "Washington", "WI": "Wisconsin"
}

segment_data["state_standardized"] = (
    segment_data["state"]
    .fillna("Unknown / Not recorded")
    .astype(str)
    .str.strip()
    .apply(
        lambda value: state_map.get(value.upper(), value)
        if value != "Unknown / Not recorded"
        else value
    )
)

# Standardise missing category labels
for column in ["membership_tier", "preferred_channel", "referral_source"]:
    segment_data[column] = (
        segment_data[column]
        .fillna("Unknown / Not recorded")
        .astype(str)
        .str.strip()
        .replace("", "Unknown / Not recorded")
    )

# Derive signup year
segment_data["signup_date"] = pd.to_datetime(
    segment_data["signup_date"], errors="coerce"
)
segment_data["signup_cohort"] = (
    segment_data["signup_date"].dt.year.astype("Int64").astype("string")
    .fillna("Unknown / Not recorded")
)

def calculate_churn_by_segment(data, segment_column):
    result = (
        data.groupby(segment_column, dropna=False)
        .agg(
            customers=("customer_id", "nunique"),
            churned_customers=("churn", "sum")
        )
        .reset_index()
    )

    result["churn_rate_pct"] = (
        result["churned_customers"] / result["customers"] * 100
    )

    return result.sort_values("churn_rate_pct", ascending=False)

segments = {
    "MEMBERSHIP TIER": "membership_tier",
    "PREFERRED CHANNEL": "preferred_channel",
    "STATE": "state_standardized",
    "REFERRAL SOURCE": "referral_source",
    "SIGNUP COHORT": "signup_cohort"
}

for title, column in segments.items():
    print(f"\n{title}")
    print(
        calculate_churn_by_segment(segment_data, column)
        .round(2)
        .to_string(index=False)
    )


MEMBERSHIP TIER
membership_tier  customers  churned_customers  churn_rate_pct
           Gold       1126                765           67.94
         Silver       1768               1184           66.97
       Platinum        411                274           66.67
         Bronze       2785               1852           66.50

PREFERRED CHANNEL
preferred_channel  customers  churned_customers  churn_rate_pct
         In-Store       2041               1388           68.01
       Mobile App       2053               1369           66.68
           Online       1996               1318           66.03

STATE
state_standardized  customers  churned_customers  churn_rate_pct
          Michigan        294                208           70.75
         Wisconsin        306                215           70.26
             Texas        298                208           69.80
              Ohio        292                200           68.49
          Illinois        314                215           68.47
 

In [21]:
# PHASE 7: SUPPORT ISSUE CATEGORY AND CUSTOMER CHURN

# Keep valid support tickets recorded before the snapshot date
tickets_for_analysis = support_tickets.copy()

tickets_for_analysis["ticket_date"] = pd.to_datetime(
    tickets_for_analysis["ticket_date"],
    errors="coerce"
)

tickets_for_analysis = tickets_for_analysis[
    tickets_for_analysis["ticket_date"].notna()
    & (tickets_for_analysis["ticket_date"] < SNAPSHOT_DATE)
    & tickets_for_analysis["customer_id"].notna()
].copy()

# Standardise issue-category labels
tickets_for_analysis["issue_category"] = (
    tickets_for_analysis["issue_category"]
    .fillna("Unknown / Not recorded")
    .astype(str)
    .str.strip()
    .replace("", "Unknown / Not recorded")
)

# Attach the corrected churn label to each ticket
tickets_with_churn = tickets_for_analysis.merge(
    customer_analysis[["customer_id", "churn"]],
    on="customer_id",
    how="inner"
)

# Count each customer only once within each issue category
customer_issue = tickets_with_churn[
    ["customer_id", "issue_category", "churn"]
].drop_duplicates()

issue_customer_summary = (
    customer_issue
    .groupby("issue_category")
    .agg(
        customers=("customer_id", "nunique"),
        churned_customers=("churn", "sum")
    )
    .reset_index()
)

issue_customer_summary["churn_rate_pct"] = (
    issue_customer_summary["churned_customers"]
    / issue_customer_summary["customers"] * 100
)

# Summarise ticket-level satisfaction and resolution time
issue_ticket_summary = (
    tickets_with_churn
    .groupby("issue_category")
    .agg(
        ticket_count=("ticket_id", "nunique"),
        avg_satisfaction=("satisfaction_score", "mean"),
        avg_resolution_hours=("resolution_time_hours", "mean")
    )
    .reset_index()
)

# Combine the customer-level and ticket-level summaries
issue_summary = issue_customer_summary.merge(
    issue_ticket_summary,
    on="issue_category",
    how="left"
)

issue_summary = issue_summary.sort_values(
    "churn_rate_pct",
    ascending=False
)

print("SUPPORT ISSUE CATEGORY AND CUSTOMER CHURN")
print(issue_summary.round(2).to_string(index=False))

SUPPORT ISSUE CATEGORY AND CUSTOMER CHURN
    issue_category  customers  churned_customers  churn_rate_pct  ticket_count  avg_satisfaction  avg_resolution_hours
  Product Question        683                472           69.11           715              3.94                 17.74
Wrong Item Shipped        566                387           68.37           601              3.93                 18.07
     Refund Status        634                433           68.30           671              3.93                 18.09
    Account Access        630                427           67.78           661              3.85                 18.93
      Damaged Item        634                429           67.67           663              3.87                 18.55
    Return Request        614                415           67.59           641              3.88                 17.79
     Billing Issue        601                406           67.55           633              3.84                 19.69
    De

In [22]:
# PHASE 7: MONTHLY COMPLETED-ORDER ACTIVITY

monthly_orders = orders.copy()

monthly_orders["order_date"] = pd.to_datetime(
    monthly_orders["order_date"],
    errors="coerce"
)

monthly_completed = monthly_orders[
    (monthly_orders["order_status"].astype(str).str.strip().str.lower()
     == "completed")
    & monthly_orders["order_date"].notna()
    & (monthly_orders["order_date"] < SNAPSHOT_DATE)
].copy()

monthly_completed["order_month"] = (
    monthly_completed["order_date"].dt.to_period("M").astype(str)
)

monthly_summary = (
    monthly_completed
    .groupby("order_month")
    .agg(
        completed_orders=("order_id", "nunique"),
        unique_customers=("customer_id", "nunique"),
        total_order_value=("total_amount", "sum"),
        average_order_value=("total_amount", "mean")
    )
    .reset_index()
    .sort_values("order_month")
)

print("MONTHLY COMPLETED-ORDER ACTIVITY")
print(monthly_summary.round(2).to_string(index=False))

print("\nDATE COVERAGE")
print("First completed-order month:", monthly_summary["order_month"].min())
print("Last completed-order month:", monthly_summary["order_month"].max())
print("Months represented:", len(monthly_summary))

MONTHLY COMPLETED-ORDER ACTIVITY
order_month  completed_orders  unique_customers  total_order_value  average_order_value
    2020-09                 4                 4            3240.49               810.12
    2020-10                 7                 6            7268.74              1038.39
    2020-11                11                10            7773.03               706.64
    2020-12                16                16           28811.11              1800.69
    2021-01                20                19           30773.47              1538.67
    2021-02                30                29           42739.52              1424.65
    2021-03                32                32           56694.86              1771.71
    2021-04                50                48           82807.29              1656.15
    2021-05                51                48           73705.03              1445.20
    2021-06                48                46           58992.03              1229.00

In [23]:
# PHASE 7: INVESTIGATE THE MAY 2025 ORDER SPIKE

orders_check = orders.copy()

orders_check["order_date"] = pd.to_datetime(
    orders_check["order_date"], errors="coerce"
)

# Focus on April to July 2025
period_start = pd.Timestamp("2025-04-01")
period_end = pd.Timestamp("2025-08-01")

period_orders = orders_check[
    orders_check["order_date"].ge(period_start)
    & orders_check["order_date"].lt(period_end)
    & (orders_check["order_status"].astype(str).str.strip().str.lower()
       == "completed")
].copy()

period_orders["order_month"] = (
    period_orders["order_date"].dt.to_period("M").astype(str)
)

# Monthly totals
monthly_check = (
    period_orders.groupby("order_month")
    .agg(
        completed_orders=("order_id", "nunique"),
        unique_customers=("customer_id", "nunique"),
        total_order_value=("total_amount", "sum"),
        average_order_value=("total_amount", "mean"),
        median_order_value=("total_amount", "median"),
        maximum_order_value=("total_amount", "max")
    )
    .reset_index()
    .sort_values("order_month")
)

print("MONTHLY CHECK: APRIL–JULY 2025")
print(monthly_check.round(2).to_string(index=False))

# Examine whether one sales channel accounts for much of the spike
channel_check = (
    period_orders.groupby(["order_month", "channel"])
    .agg(
        completed_orders=("order_id", "nunique"),
        unique_customers=("customer_id", "nunique"),
        total_order_value=("total_amount", "sum")
    )
    .reset_index()
    .sort_values(["order_month", "completed_orders"],
                 ascending=[True, False])
)

print("\nCOMPLETED ORDERS BY MONTH AND CHANNEL")
print(channel_check.round(2).to_string(index=False))

MONTHLY CHECK: APRIL–JULY 2025
order_month  completed_orders  unique_customers  total_order_value  average_order_value  median_order_value  maximum_order_value
    2025-04               556               461          886272.53              1594.02             1352.22              8164.26
    2025-05              1102               635         1704178.43              1546.44             1253.48              8826.77
    2025-06               902               643         1416504.21              1570.40             1272.01              7594.68
    2025-07               784               607         1151995.74              1469.38             1180.34              6277.87

COMPLETED ORDERS BY MONTH AND CHANNEL
order_month    channel  completed_orders  unique_customers  total_order_value
    2025-04     Online               191               184          298674.84
    2025-04 Mobile App               186               167          284713.71
    2025-04   In-Store               179           

In [24]:
# PHASE 7: ORDERS PER CUSTOMER, APRIL–JULY 2025

customer_month_activity = (
    period_orders
    .groupby(["order_month", "customer_id"])
    .agg(
        completed_orders=("order_id", "nunique"),
        total_order_value=("total_amount", "sum")
    )
    .reset_index()
)

orders_per_customer_summary = (
    customer_month_activity
    .groupby("order_month")
    .agg(
        unique_customers=("customer_id", "nunique"),
        avg_orders_per_customer=("completed_orders", "mean"),
        median_orders_per_customer=("completed_orders", "median"),
        customers_with_multiple_orders=(
            "completed_orders",
            lambda x: (x >= 2).sum()
        ),
        avg_spend_per_customer=("total_order_value", "mean")
    )
    .reset_index()
    .sort_values("order_month")
)

orders_per_customer_summary["multiple_order_customer_pct"] = (
    orders_per_customer_summary["customers_with_multiple_orders"]
    / orders_per_customer_summary["unique_customers"] * 100
)

print("CUSTOMER ORDER FREQUENCY: APRIL–JULY 2025")
print(orders_per_customer_summary.round(2).to_string(index=False))

CUSTOMER ORDER FREQUENCY: APRIL–JULY 2025
order_month  unique_customers  avg_orders_per_customer  median_orders_per_customer  customers_with_multiple_orders  avg_spend_per_customer  multiple_order_customer_pct
    2025-04               461                     1.21                         1.0                              58                 1922.50                        12.58
    2025-05               635                     1.74                         1.0                             182                 2683.75                        28.66
    2025-06               643                     1.40                         1.0                             141                 2202.96                        21.93
    2025-07               607                     1.29                         1.0                             116                 1897.85                        19.11


A closer examination of Meridian Retail Group's May 2025 transaction spike showed that unique purchasing customers increased by approximately 37.7% compared with April, while the proportion of purchasing customers placing multiple orders within the month rose from 12.58% to 28.66%. Average orders per purchasing customer increased from 1.21 to 1.74 before declining in June and July. These results suggest that the May spike involved both broader purchasing participation and more frequent within-month ordering. However, the available analysis does not establish the cause of the increase or demonstrate that the activity translated into long-term retention.